In [ ]:
# Example 1: Create an in-memory database and a table via fastlite.database.
from fastlite import database

db = database(':memory:')
print(f"db type: {type(db).__name__}")
print(f"initial tables: {db.t}")

# Create a table by passing a schema dict.
users = db.t.users
users.create(id=int, name=str, age=int, pk='id')
print(f"created table: {users}")
print(f"columns: {[c.name for c in users.columns]}")
assert 'users' in [t.name for t in db.t]


In [ ]:
# Example 2: Insert rows and fetch them back.
from fastlite import database

db = database(':memory:')
users = db.t.users
users.create(id=int, name=str, age=int, pk='id')

users.insert(dict(id=1, name='Ada', age=36))
users.insert(dict(id=2, name='Grace', age=85))
users.insert(dict(id=3, name='Linus', age=54))

rows = list(users())
print(f"row count: {len(rows)}")
for r in rows:
    print(f"  {r}")
assert len(rows) == 3
assert any(r['name'] == 'Ada' for r in rows)


In [ ]:
# Example 3: Query with where-clause + ordering.
from fastlite import database

db = database(':memory:')
users = db.t.users
users.create(id=int, name=str, age=int, pk='id')
for i, (n, a) in enumerate([('Ada', 36), ('Grace', 85), ('Linus', 54), ('Bjarne', 73)], start=1):
    users.insert(dict(id=i, name=n, age=a))

older = list(users(where='age > 50', order_by='age'))
names = [r['name'] for r in older]
print(f"older than 50, sorted by age: {names}")
assert names == ['Linus', 'Bjarne', 'Grace']


In [ ]:
# Example 4: dataclass-style Row objects via .dataclass().
from fastlite import database

db = database(':memory:')
users = db.t.users
users.create(id=int, name=str, age=int, pk='id')
users.insert(dict(id=1, name='Ada', age=36))
users.insert(dict(id=2, name='Grace', age=85))

User = users.dataclass()
print(f"dataclass name: {User.__name__}")
print(f"dataclass fields: {list(User.__dataclass_fields__.keys())}")

# After .dataclass(), fetches return dataclass instances (attribute access, not dicts).
fetched = users(where='id = 1')[0]
print(f"row object: {fetched}")
assert isinstance(fetched, User)
assert fetched.name == 'Ada' and fetched.age == 36


In [ ]:
# Example 5: Raw SQL via db.q and table update/delete.
from fastlite import database

db = database(':memory:')
users = db.t.users
users.create(id=int, name=str, age=int, pk='id')
for i, (n, a) in enumerate([('Ada', 36), ('Grace', 85), ('Linus', 54)], start=1):
    users.insert(dict(id=i, name=n, age=a))

# Raw SQL aggregation.
result = db.q('SELECT COUNT(*) AS c, AVG(age) AS avg_age FROM users')
print(f"aggregate: {result}")

# Update + delete through the table helper.
users.update(dict(id=1, age=37))
users.delete(2)
remaining = list(users())
print(f"after update+delete: {remaining}")
assert len(remaining) == 2
assert next(r for r in remaining if r['id'] == 1)['age'] == 37
